# Cài thư viện + Setup

In [1]:
!pip install -q pyvi emoji transformers scikit-learn openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 47.4 MB/s eta 0:00:00


In [2]:


import os
import json
import pickle
import time
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_scheduler
from sklearn.metrics import f1_score, classification_report
from pyvi.ViTokenizer import tokenize
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

os.makedirs("/kaggle/working/saved_models", exist_ok=True)
os.makedirs("/kaggle/working/reports", exist_ok=True)
os.makedirs("/kaggle/working/corpus", exist_ok=True)

Device: cuda


# Load dữ liệu 

In [3]:
import subprocess

REPO_URL = "https://github.com/ricardo-tran/ViGoEmotions.git"
REPO_DIR = "/kaggle/working/ViGoEmotions_Original"

if not os.path.exists(REPO_DIR):
    print("--- Đang clone repo ---")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print("--- Repo đã tồn tại ---")

DOCS_PATH = os.path.join(REPO_DIR, "model", "docs")
CORPUS_PATH = os.path.join(REPO_DIR, "corpus")

print("DOCS_PATH  :", DOCS_PATH)
print("CORPUS_PATH:", CORPUS_PATH)

# Kiểm tra xem có file pickle S3 không
pkl_candidates = [
    os.path.join(CORPUS_PATH, "s3_visolex_train-val-test.pkl"),
    os.path.join(REPO_DIR, "corpus", "s3_visolex_train-val-test.pkl"),
]

pkl_path = None
for p in pkl_candidates:
    if os.path.exists(p):
        pkl_path = p
        break

if pkl_path:
    print(f" Tìm thấy pickle S3: {pkl_path}")
    with open(pkl_path, "rb") as f:
        train_df, val_df, test_df = pickle.load(f)
else:
    print(" Không thấy s3_visolex_train-val-test.pkl")
    print("→ Tạm dùng dataset_V1.xlsx (giống S2) + tiền xử lý S3 (chỉ lower)")
    excel_path = os.path.join(CORPUS_PATH, "dataset_V1.xlsx")
    excel_file = pd.ExcelFile(excel_path)

    if "train" in excel_file.sheet_names:
        train_df = pd.read_excel(excel_file, sheet_name="train")
        val_df   = pd.read_excel(excel_file, sheet_name="val")
        test_df  = pd.read_excel(excel_file, sheet_name="test")
    else:
        df = pd.read_excel(excel_file, sheet_name="Sheet1")
        train_df = df[df["set"] == "train"].copy()
        val_df   = df[df["set"] == "val"].copy()
        test_df  = df[df["set"] == "test"].copy()

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)
display(train_df.head(3))

--- Đang clone repo ---


Cloning into '/kaggle/working/ViGoEmotions_Original'...


DOCS_PATH  : /kaggle/working/ViGoEmotions_Original/model/docs
CORPUS_PATH: /kaggle/working/ViGoEmotions_Original/corpus
 Không thấy s3_visolex_train-val-test.pkl
→ Tạm dùng dataset_V1.xlsx (giống S2) + tiền xử lý S3 (chỉ lower)
Train: (16531, 3)
Val  : (2066, 3)
Test : (2067, 3)


,id,text,labels
0,tik000008,Xem mà ngẫm lại cuộc đời bản thân ta đã trải q...,[12]
1,5743,bức ảnh xuất sắc ❤️,"[2, 8, 3]"
2,32895,"Vừa đẹp trai, vừa tài giỏi. Nhà mặt phố, bố là...","[8, 7]"


# Tiền xử lý

In [4]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    return text.lower()

print("Applying S3 preprocessing (only lower)...")
for df in [train_df, val_df, test_df]:
    df["text"] = df["text"].astype(str).apply(clean_text)

print("Done")
print(train_df["text"].iloc[0])

Applying S3 preprocessing (only lower)...
Done
xem mà ngẫm lại cuộc đời bản thân ta đã trải qua nhiều thứ ta rồi cũng sẽ lớn kí ước sẽ còn mãi trong lòng


# Load label dict + Encode labels

In [5]:
# Ưu tiên lấy từ model/docs, nếu không có thì lấy từ corpus
label_dict_path = os.path.join(DOCS_PATH, "label_dict.json")
if not os.path.exists(label_dict_path):
    label_dict_path = os.path.join(CORPUS_PATH, "label_dict.json")

with open(label_dict_path, "r", encoding="utf-8") as f:
    label_dict = json.load(f)

label_to_idx = {label: int(idx) for idx, label in label_dict.items()}
print("Number of labels:", len(label_dict))
print("label_dict path:", label_dict_path)

def encode_labels(label, label_dict):
    labels = str(label).replace("[", "").replace("]", "").replace("'", "").replace('"', "").split(",")
    labels = [x.strip() for x in labels if x.strip()]
    vec = np.zeros(len(label_dict), dtype=np.float32)

    if labels and labels[0].isnumeric():
        labels = [int(x) for x in labels]
        for idx in label_dict.values():
            if idx in labels:
                vec[idx] = 1.0
    else:
        for lab, idx in label_dict.items():
            if lab in labels:
                vec[idx] = 1.0
    return vec

train_texts  = train_df["text"].tolist()
train_labels = [encode_labels(x, label_to_idx) for x in train_df["labels"]]
val_texts    = val_df["text"].tolist()
val_labels   = [encode_labels(x, label_to_idx) for x in val_df["labels"]]
test_texts   = test_df["text"].tolist()
test_labels  = [encode_labels(x, label_to_idx) for x in test_df["labels"]]

print("Labels encoded")

Number of labels: 28
label_dict path: /kaggle/working/ViGoEmotions_Original/model/docs/label_dict.json
Labels encoded


# Dataset + DataLoader

In [6]:
model_type = "mbert"
model_name = "google-bert/bert-base-multilingual-cased"
max_len = 200
BATCH_SIZE = 16            

tokenizer = AutoTokenizer.from_pretrained(model_name)

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = torch.tensor(labels, dtype=torch.float32)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        
        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt",
            return_attention_mask=True,
        )
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "targets": self.labels[idx],
            "text": text,
        }

train_dataset = SentimentDataset(train_texts, train_labels, tokenizer, max_len)
val_dataset   = SentimentDataset(val_texts, val_labels, tokenizer, max_len)
test_dataset  = SentimentDataset(test_texts, test_labels, tokenizer, max_len)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("DataLoader ready (mBERT)")
print("Train batches:", len(train_loader))

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

DataLoader ready (mBERT)
Train batches: 1034


/tmp/ipykernel_24/3251346832.py:11: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  self.labels = torch.tensor(labels, dtype=torch.float32)


# model

In [7]:
class ModelSentimentClassifier(nn.Module):
    def __init__(self, n_classes, model_name, model_type):
        super().__init__()
        self.model_type = model_type
        config = AutoConfig.from_pretrained(
            model_name,
            hidden_dropout_prob=0.1,
            attention_probs_dropout_prob=0.1
        )
        self.backbone = AutoModel.from_pretrained(model_name, config=config)
        self.drop = nn.Dropout(0.2)
        self.fc = nn.Linear(self.backbone.config.hidden_size, n_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        
        # mBERT thuộc họ BERT → dùng pooler_output
        if "bartpho" in self.model_type:
            pooled = outputs.last_hidden_state[:, 0, :]
        else:
            pooled = outputs.pooler_output if outputs.pooler_output is not None else outputs.last_hidden_state[:, 0, :]
        
        x = self.drop(pooled)
        return {"logits": self.fc(x)}

model = ModelSentimentClassifier(
    n_classes=len(label_dict),
    model_name=model_name,
    model_type=model_type
).to(device)

print("mBERT model loaded")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google-bert/bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


mBERT model loaded
Parameters: 177,874,972


# Train + Lưu metrics

In [8]:
EPOCHS = 12
optimizer = AdamW(model.parameters(), lr=5e-5)
scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=len(train_loader),
    num_training_steps=len(train_loader) * EPOCHS
)

label_counts = np.sum(train_labels, axis=0)
pos_weight = torch.tensor(
    [(len(train_labels) - c) / max(c, 1) for c in label_counts],
    dtype=torch.float32
).to(device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

def run_epoch(model, loader, is_train=True):
    model.train() if is_train else model.eval()
    losses, all_y, all_p = [], [], []

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for batch in tqdm(loader, leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            targets = batch["targets"].to(device)

            if is_train:
                optimizer.zero_grad()

            logits = model(input_ids, attention_mask)["logits"]
            loss = loss_fn(logits, targets)
            losses.append(loss.item())

            if is_train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()

            preds = (torch.sigmoid(logits) >= 0.5).int()
            all_y.append(targets.cpu().numpy())
            all_p.append(preds.cpu().numpy())

    y = np.vstack(all_y)
    p = np.vstack(all_p)
    f1 = f1_score(y, p, average="macro", zero_division=0)
    return np.mean(losses), f1

best_f1 = 0.0
history = {"epoch": [], "train_loss": [], "train_f1": [], "val_loss": [], "val_f1": []}

print("Start training BARTpho (S3)...")
for epoch in range(1, EPOCHS + 1):
    print(f"\n===== Epoch {epoch}/{EPOCHS} =====")
    train_loss, train_f1 = run_epoch(model, train_loader, is_train=True)
    val_loss, val_f1 = run_epoch(model, val_loader, is_train=False)

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["train_f1"].append(train_f1)
    history["val_loss"].append(val_loss)
    history["val_f1"].append(val_f1)

    print(f"Train Loss: {train_loss:.4f} | Train Macro-F1: {train_f1:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Macro-F1: {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), f"/kaggle/working/saved_models/{model_type}_s3_best.pth")
        print(f"Saved best model (Val F1 = {best_f1:.4f})")

# Lưu metrics
pd.DataFrame(history).to_excel(f"/kaggle/working/reports/metrics_{model_type}_s3.xlsx", index=False)
print("\nTraining finished!")
print(f"Metrics saved: /kaggle/working/reports/metrics_{model_type}_s3.xlsx")

Start training BARTpho (S3)...

===== Epoch 1/12 =====


Train Loss: 1.1172 | Train Macro-F1: 0.2007
Val   Loss: 0.9427 | Val   Macro-F1: 0.2847
Saved best model (Val F1 = 0.2847)

===== Epoch 2/12 =====


Train Loss: 0.8686 | Train Macro-F1: 0.3218
Val   Loss: 0.8463 | Val   Macro-F1: 0.3628
Saved best model (Val F1 = 0.3628)

===== Epoch 3/12 =====


Train Loss: 0.7101 | Train Macro-F1: 0.3962
Val   Loss: 0.8061 | Val   Macro-F1: 0.3963
Saved best model (Val F1 = 0.3963)

===== Epoch 4/12 =====


Train Loss: 0.5741 | Train Macro-F1: 0.4624
Val   Loss: 0.8440 | Val   Macro-F1: 0.4238
Saved best model (Val F1 = 0.4238)

===== Epoch 5/12 =====


...

Train Loss: 0.2714 | Train Macro-F1: 0.6865
Val   Loss: 1.1005 | Val   Macro-F1: 0.4855
Saved best model (Val F1 = 0.4855)

===== Epoch 9/12 =====


Train Loss: 0.2280 | Train Macro-F1: 0.7320
Val   Loss: 1.1526 | Val   Macro-F1: 0.4928
Saved best model (Val F1 = 0.4928)

===== Epoch 10/12 =====


Train Loss: 0.1926 | Train Macro-F1: 0.7695
Val   Loss: 1.2243 | Val   Macro-F1: 0.4972
Saved best model (Val F1 = 0.4972)

===== Epoch 11/12 =====


Train Loss: 0.1627 | Train Macro-F1: 0.8045
Val   Loss: 1.2646 | Val   Macro-F1: 0.4978
Saved best model (Val F1 = 0.4978)

===== Epoch 12/12 =====


Train Loss: 0.1445 | Train Macro-F1: 0.8258
Val   Loss: 1.3253 | Val   Macro-F1: 0.5044
Saved best model (Val F1 = 0.5044)

Training finished!
Metrics saved: /kaggle/working/reports/metrics_mbert_s3.xlsx


# Test + Lưu Classification Report

In [9]:
model.load_state_dict(torch.load(f"/kaggle/working/saved_models/{model_type}_s3_best.pth"))
model.eval()

all_targets, all_preds = [], []
with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        targets = batch["targets"].to(device)

        logits = model(input_ids, attention_mask)["logits"]
        preds = (torch.sigmoid(logits) >= 0.5).int()

        all_targets.append(targets.cpu().numpy())
        all_preds.append(preds.cpu().numpy())

y_true = np.vstack(all_targets)
y_pred = np.vstack(all_preds)

macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)

print("\n TEST RESULTS (BARTpho - S3) ")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Micro F1: {micro_f1:.4f}")

report = classification_report(
    y_true, y_pred,
    target_names=list(label_dict.values()),
    zero_division=0,
    output_dict=True
)
pd.DataFrame(report).transpose().to_excel(
    f"/kaggle/working/reports/classification_report_{model_type}_s3.xlsx",
    index=True
)

print(classification_report(y_true, y_pred, target_names=list(label_dict.values()), zero_division=0))
print(f"\nReport saved: /kaggle/working/reports/classification_report_{model_type}_s3.xlsx")

100%|██████████| 130/130 [00:28<00:00,  4.55it/s]


 TEST RESULTS (BARTpho - S3) 
Macro F1: 0.5104
Micro F1: 0.5278
                precision    recall  f1-score   support

     amusement       0.54      0.84      0.66       374
    excitement       0.28      0.42      0.34        98
           joy       0.39      0.64      0.49       204
          love       0.43      0.71      0.53       143
        desire       0.35      0.54      0.42        80
      optimism       0.53      0.78      0.63       142
        caring       0.49      0.73      0.58       150
         pride       0.50      0.65      0.56        86
    admiration       0.42      0.55      0.48       101
     gratitude       0.68      0.88      0.77       108
        relief       0.30      0.58      0.39        60
      approval       0.41      0.66      0.51       115
   realization       0.27      0.42      0.33        95
      surprise       0.39      0.54      0.46        85
     curiosity       0.47      0.67      0.55       100
     confusion       0.35      0.49   